In [1]:
!pip install -q transformers datasets scikit-learn torch matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 31.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, hamming_loss
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_scheduler
from tqdm.notebook import tqdm
import torch.nn.functional as F


In [3]:
import pandas as pd

df = pd.read_csv("track-a.csv")
print(df.head())
print(df.columns)


                        id                                               text  \
0  eng_train_track_a_00001                       Colorado, middle of nowhere.   
1  eng_train_track_a_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_a_00003        It was one of my most shameful experiences.   
3  eng_train_track_a_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_a_00005                        Then the screaming started.   

   anger  fear  joy  sadness  surprise  
0      0     1    0        0         1  
1      0     1    0        0         0  
2      0     1    0        1         0  
3      0     0    0        0         0  
4      0     1    0        1         1  
Index(['id', 'text', 'anger', 'fear', 'joy', 'sadness', 'surprise'], dtype='object')


In [4]:
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer

# Set seed for reproducibility
np.random.seed(42)

# Define features and labels
texts = df["text"].values
labels = df[["anger", "fear", "joy", "sadness", "surprise"]].values

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.2, random_state=42)

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize (returns input_ids and attention_mask)
def tokenize_texts(texts, max_len=128):
    return tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )

train_encodings = tokenize_texts(X_train)
val_encodings = tokenize_texts(X_val)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [5]:
from torch.utils.data import Dataset, DataLoader

class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels).float()  # float for BCE loss

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# Create dataset objects
train_dataset = EmotionDataset(train_encodings, y_train)
val_dataset = EmotionDataset(val_encodings, y_val)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


In [6]:
from transformers import BertForSequenceClassification

# 5 emotion labels
num_labels = 5

# Load pre-trained BERT for sequence classification
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
from torch.optim import AdamW
from transformers import get_scheduler

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer & scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 4
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)


In [8]:
from tqdm.notebook import tqdm

model.train()
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    total_loss = 0

    loop = tqdm(train_loader, leave=False)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        loop.set_description(f"Epoch {epoch + 1}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Training Loss: {avg_loss:.4f}")



Epoch 1/4


  0%|          | 0/139 [00:00<?, ?it/s]

Training Loss: 0.5185

Epoch 2/4


  0%|          | 0/139 [00:00<?, ?it/s]

Training Loss: 0.3751

Epoch 3/4


  0%|          | 0/139 [00:00<?, ?it/s]

Training Loss: 0.2887

Epoch 4/4


  0%|          | 0/139 [00:00<?, ?it/s]

Training Loss: 0.2439


In [9]:
from sklearn.metrics import f1_score, hamming_loss, classification_report

model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        probs = torch.sigmoid(logits)  # convert to probabilities
        predictions.append(probs.cpu().numpy())
        true_labels.append(batch["labels"].cpu().numpy())

# Stack all predictions and labels
predictions = np.vstack(predictions)
true_labels = np.vstack(true_labels)

# Apply threshold to get binary predictions
pred_labels = (predictions >= 0.5).astype(int)

# Micro-F1 score
micro_f1 = f1_score(true_labels, pred_labels, average='micro')
print(f"🔹 Micro-F1 score: {micro_f1:.4f}")

# Hamming Loss
hamming = hamming_loss(true_labels, pred_labels)
print(f"🔹 Hamming Loss: {hamming:.4f}")

# Label-wise report
emotion_labels = ["anger", "fear", "joy", "sadness", "surprise"]
print("\n🔹 Classification Report:")
print(classification_report(true_labels, pred_labels, target_names=emotion_labels))


🔹 Micro-F1 score: 0.7277
🔹 Hamming Loss: 0.1632

🔹 Classification Report:
              precision    recall  f1-score   support

       anger       0.82      0.38      0.51        72
        fear       0.79      0.84      0.81       330
         joy       0.67      0.62      0.64       115
     sadness       0.69      0.63      0.66       167
    surprise       0.80      0.70      0.74       179

   micro avg       0.76      0.70      0.73       863
   macro avg       0.75      0.63      0.67       863
weighted avg       0.76      0.70      0.72       863
 samples avg       0.66      0.64      0.63       863



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# Save model and tokenizer
model.save_pretrained("emotion-bert-model")
tokenizer.save_pretrained("emotion-bert-model")


('emotion-bert-model/tokenizer_config.json',
 'emotion-bert-model/special_tokens_map.json',
 'emotion-bert-model/vocab.txt',
 'emotion-bert-model/added_tokens.json')

In [11]:
def predict_emotions(text, threshold=0.5):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    # Convert probs to binary predictions
    binary = (probs >= threshold).astype(int)
    emotions = ["anger", "fear", "joy", "sadness", "surprise"]

    # Create readable output
    result = {emotion: f"{prob:.2f}" for emotion, prob in zip(emotions, probs)}
    detected = [e for e, b in zip(emotions, binary) if b == 1]

    print("Predicted emotions (threshold ≥ 0.5):", detected)
    print("Probabilities:", result)

# Example
predict_emotions("I can't believe this happened!")


Predicted emotions (threshold ≥ 0.5): ['fear', 'surprise']
Probabilities: {'anger': '0.16', 'fear': '0.86', 'joy': '0.16', 'sadness': '0.14', 'surprise': '0.92'}


In [12]:
# Interactive input in Colab
text = input("Enter a sentence to detect emotions: ")
predict_emotions(text)


Enter a sentence to detect emotions: this is pure hatred
Predicted emotions (threshold ≥ 0.5): ['anger', 'fear', 'sadness']
Probabilities: {'anger': '0.86', 'fear': '0.70', 'joy': '0.06', 'sadness': '0.64', 'surprise': '0.38'}


In [13]:
from sklearn.metrics import coverage_error

# true_labels: shape [n_samples, n_classes]
# predictions: predicted probabilities, same shape

coverage = coverage_error(true_labels, predictions)
print(f"🔹 Coverage Error: {coverage:.4f}")


🔹 Coverage Error: 1.9874


In [14]:
def predict_emotions(text, threshold=0.5):
    """
    Predict emotions from input text using the fine-tuned BERT model.

    Returns:
        probs: list of emotion probabilities
        binary: list of 0/1 predictions after thresholding
        labels: emotion label names
    """
    model.eval()
    emotion_labels = ["anger", "fear", "joy", "sadness", "surprise"]

    # Tokenize and predict
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    # Binary predictions based on threshold
    binary = (probs >= threshold).astype(int)

    # Display predictions
    predicted = [emotion_labels[i] for i, val in enumerate(binary) if val == 1]
    print(f"\n🔹 Input: {text}")
    print(f"🔹 Predicted emotions (≥ {threshold}): {predicted}")
    print(f"🔹 Probabilities: {{", end="")
    print(", ".join([f"{emotion}: {probs[i]:.2f}" for i, emotion in enumerate(emotion_labels)]), end="}}\n")

    return probs, binary, emotion_labels


In [15]:
# Step 1: Enter your sentence
text = input("Enter a sentence to analyze: ")

# Step 2: Get predictions
probs, binary, labels = predict_emotions(text, threshold=0.5)

# Step 3: Define expected (true) emotions manually for simulation
expected = ["anger", "sadness"]  # CHANGE THIS based on what you think is true
true = [1 if label in expected else 0 for label in labels]

# Step 4: Simulate Coverage Error
sorted_indices = np.argsort(probs)[::-1]
sorted_labels = [labels[i] for i in sorted_indices]

true_indices = set(i for i, v in enumerate(true) if v == 1)
covered = set()
coverage_error_example = len(labels)

for k, idx in enumerate(sorted_indices):
    if idx in true_indices:
        covered.add(idx)
    if covered == true_indices:
        coverage_error_example = k + 1
        break

print(f"\n🧮 Simulated Coverage Error for this sentence: {coverage_error_example}")
print(f"🏷️ Top-ranked labels: {sorted_labels[:coverage_error_example]}")


Enter a sentence to analyze: this is pure hatred

🔹 Input: this is pure hatred
🔹 Predicted emotions (≥ 0.5): ['anger', 'fear', 'sadness']
🔹 Probabilities: {anger: 0.86, fear: 0.70, joy: 0.06, sadness: 0.64, surprise: 0.38}}

🧮 Simulated Coverage Error for this sentence: 3
🏷️ Top-ranked labels: ['anger', 'fear', 'sadness']


In [16]:
from sklearn.metrics import accuracy_score

# Exact Match (subset accuracy)
exact_match = accuracy_score(true_labels, pred_labels)
print(f"🔹 Exact Match Accuracy: {exact_match:.4f}")


🔹 Exact Match Accuracy: 0.4061


In [17]:
# Save model and tokenizer locally
model.save_pretrained("saved_model")
tokenizer.save_pretrained("saved_model")


('saved_model/tokenizer_config.json',
 'saved_model/special_tokens_map.json',
 'saved_model/vocab.txt',
 'saved_model/added_tokens.json')

In [18]:
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification

# Define local model folder
model_path = "saved_model"  # Folder must contain model files

# Load model and tokenizer from local directory (offline mode)
try:
    model = BertForSequenceClassification.from_pretrained(model_path, local_files_only=True)
    tokenizer = BertTokenizer.from_pretrained(model_path, local_files_only=True)
except Exception as e:
    print(f"❌ Error loading model from '{model_path}'.")
    print("Make sure you ran model.save_pretrained('saved_model') and tokenizer.save_pretrained('saved_model')")
    raise e  # Don't use sys.exit() in Colab, this is safer

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Define emotion labels (must match your training order)
emotion_labels = ["anger", "fear", "joy", "sadness", "surprise"]

def predict(csv_path):
    """
    Predict emotions for each text in the input CSV.

    Args:
        csv_path (str): Path to CSV file with a 'text' column.

    Returns:
        List[List[int]]: Binary predictions for each emotion label.
    """
    try:
        df = pd.read_csv("track-a.csv")
    except Exception as e:
        raise ValueError(f"❌ Failed to read CSV: {e}")

    if "text" not in df.columns:
        raise ValueError("❌ CSV must contain a column named 'text'.")

    predictions = []

    for text in df["text"]:
        # Tokenize text
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Predict
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.sigmoid(logits).cpu().numpy()[0]
            binary = (probs >= 0.5).astype(int).tolist()
            predictions.append(binary)

    return predictions

# Optional CLI usage
if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("Usage: python main.py path/to/input.csv")
    else:
        result = predict(sys.argv[1])
        for i, row in enumerate(result):
            print(f"{i + 1}: {row}")


1: [0, 1, 0, 0, 1]
2: [0, 1, 0, 0, 0]
3: [0, 1, 0, 1, 0]
4: [0, 0, 1, 0, 0]
5: [0, 1, 0, 1, 1]
6: [0, 1, 0, 0, 1]
7: [0, 1, 0, 0, 0]
8: [0, 1, 0, 1, 0]
9: [0, 1, 0, 0, 0]
10: [0, 0, 1, 0, 0]
11: [0, 0, 0, 0, 1]
12: [0, 0, 1, 0, 1]
13: [0, 0, 1, 0, 0]
14: [0, 0, 0, 0, 0]
15: [0, 0, 1, 0, 0]
16: [0, 1, 0, 0, 1]
17: [0, 1, 0, 1, 0]
18: [0, 0, 1, 0, 0]
19: [0, 1, 0, 0, 1]
20: [0, 1, 0, 0, 1]
21: [0, 0, 1, 0, 0]
22: [0, 0, 1, 0, 0]
23: [0, 0, 0, 0, 0]
24: [0, 1, 0, 0, 1]
25: [0, 1, 0, 1, 0]
26: [0, 1, 0, 0, 1]
27: [1, 1, 0, 0, 0]
28: [0, 1, 0, 0, 0]
29: [0, 1, 0, 0, 0]
30: [0, 1, 0, 0, 1]
31: [0, 1, 0, 0, 0]
32: [0, 0, 1, 0, 0]
33: [0, 0, 1, 0, 0]
34: [0, 1, 0, 0, 0]
35: [0, 1, 0, 0, 1]
36: [0, 1, 0, 1, 0]
37: [1, 1, 0, 0, 1]
38: [0, 1, 0, 0, 0]
39: [1, 1, 0, 1, 0]
40: [0, 1, 0, 0, 1]
41: [0, 0, 1, 1, 0]
42: [0, 1, 0, 1, 0]
43: [0, 0, 1, 0, 1]
44: [0, 0, 0, 0, 0]
45: [0, 0, 1, 0, 0]
46: [0, 1, 0, 1, 0]
47: [0, 1, 0, 1, 0]
48: [0, 1, 0, 0, 1]
49: [0, 1, 0, 0, 0]
50: [0, 0, 1, 0, 0]
51: [0, 0

In [19]:
!ls saved_model


config.json	   special_tokens_map.json  vocab.txt
model.safetensors  tokenizer_config.json
